# Null Loop Experiment Analysis

This notebook analyzes the results from our null loop experiments comparing base vs instruct models.

## Experiment Overview
- **Base Model**: Llama-3-8B.Q4_K_M.gguf
- **Instruct Model**: Llama-3-8B-Instruct.Q4_K_M.gguf
- **Seeds**: 0-19 (20 seeds each)
- **Steps**: 20 steps per seed
- **Metrics**: SSR (Self-Starting Reasoning), TIAR (Tool Invocation Attempt Rate), SRV (Self-Referential Termination)

## Analysis Goals
1. **Base Models**: Quantify "steps to EOF" - when does it start consistently looping EOF language?
2. **Instruct Models**: Classify terminal behaviors and count turns to steady-state
3. **Comparative Analysis**: Visualize differences between base and instruct models


## Setup & Imports


In [1]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import re
from collections import Counter
from typing import Dict, List, Tuple, Optional

# Set up plotting style
plt.style.use('default')
sns.set_palette("husl")

# Data paths
BASE_RESULTS_DIR = Path("results_base")
INSTRUCT_RESULTS_DIR = Path("results_instruct")

print("Setup complete!")


Matplotlib is building the font cache; this may take a moment.


Setup complete!


## Data Loading Functions


In [3]:
def load_experiment_results(results_dir: Path) -> Dict[int, Dict]:
    """Load all JSON results from a directory.
    
    Returns:
        Dict mapping seed number to experiment results
    """
    results = {}
    
    for json_file in sorted(results_dir.glob("seed_*.json")):
        # Extract seed number from filename
        seed_match = re.search(r'seed_(\d+)\.json', json_file.name)
        if seed_match:
            seed = int(seed_match.group(1))
            with open(json_file, 'r') as f:
                results[seed] = json.load(f)
    
    return results

def create_summary_dataframe(results: Dict[int, Dict]) -> pd.DataFrame:
    """Create a summary DataFrame from experiment results."""
    data = []
    
    for seed, result in results.items():
        data.append({
            'seed': seed,
            'SSR': result['SSR'],
            'TIAR': result['TIAR'], 
            'SRV': result['SRV'],
            'n_steps': len(result['steps']),
            'steps': result['steps']  # Keep raw steps for analysis
        })
    
    return pd.DataFrame(data).sort_values('seed')

# Load the data
base_results = load_experiment_results(BASE_RESULTS_DIR)
instruct_results = load_experiment_results(INSTRUCT_RESULTS_DIR)

base_df = create_summary_dataframe(base_results)
instruct_df = create_summary_dataframe(instruct_results)

print(f"Loaded {len(base_results)} base results and {len(instruct_results)} instruct results")
print(f"Base model summary:")
print(base_df[['seed', 'SSR', 'TIAR', 'SRV', 'n_steps']].head())
print(f"\nInstruct model summary:")
print(instruct_df[['seed', 'SSR', 'TIAR', 'SRV', 'n_steps']].head())


Loaded 20 base results and 20 instruct results
Base model summary:
    seed  SSR  TIAR  SRV  n_steps
0      0    0     0    0        3
1      1    0     0    0       20
12     2    0     0    0       20
13     3    0     0    0       20
14     4    0     0    0       20

Instruct model summary:
    seed  SSR  TIAR  SRV  n_steps
0      0    0     0    0       20
1      1    0     0    0       20
12     2    1     0    0       20
13     3    1     1    0       20
14     4    1     0    0       20


## Base Model Analysis: Steps to EOF


In [5]:
def analyze_eof_patterns(steps: List[str]) -> Dict:
    """Analyze EOF patterns in base model output.
    
    Looks for when the model reaches a stable EOF state (consistent EOF messages).
    
    Returns:
        Dict with EOF analysis metrics
    """
    # Look for EOF-related patterns
    eof_patterns = [
        r'>\s*EOF by user',
        r'>\s*EOF',
        r'EOF by user',
        r'end of file',
    ]
    
    eof_occurrences = []
    first_eof_step = None
    
    for i, step in enumerate(steps):
        step_clean = step.strip()
        
        # Check for EOF patterns
        for pattern in eof_patterns:
            if re.search(pattern, step_clean, re.IGNORECASE):
                eof_occurrences.append(i)
                if first_eof_step is None:
                    first_eof_step = i
                break
    
    # Find when it reaches stable EOF state (consecutive EOF messages)
    stable_eof_step = None
    if len(eof_occurrences) >= 2:
        # Look for 3+ consecutive EOF steps as "stable state"
        for i in range(len(eof_occurrences) - 2):
            if (eof_occurrences[i+1] == eof_occurrences[i] + 1 and 
                eof_occurrences[i+2] == eof_occurrences[i] + 2):
                stable_eof_step = eof_occurrences[i]
                break
    
    # Calculate metrics
    total_eof_steps = len(eof_occurrences)
    eof_frequency = total_eof_steps / len(steps) if steps else 0
    
    # Check for EOF loops (consecutive EOF steps)
    eof_loops = 0
    if len(eof_occurrences) > 1:
        for i in range(1, len(eof_occurrences)):
            if eof_occurrences[i] == eof_occurrences[i-1] + 1:
                eof_loops += 1
    
    return {
        'first_eof_step': first_eof_step,
        'stable_eof_step': stable_eof_step,
        'total_eof_steps': total_eof_steps,
        'eof_frequency': eof_frequency,
        'eof_loops': eof_loops,
        'eof_occurrences': eof_occurrences
    }

# Analyze base model EOF patterns
base_eof_analysis = []

for _, row in base_df.iterrows():
    eof_analysis = analyze_eof_patterns(row['steps'])
    eof_analysis['seed'] = row['seed']
    base_eof_analysis.append(eof_analysis)

base_eof_df = pd.DataFrame(base_eof_analysis)

print("Base Model EOF Analysis:")
print(base_eof_df[['seed', 'first_eof_step', 'stable_eof_step', 'total_eof_steps', 'eof_frequency']].head(10))

# Summary statistics
print(f"\nBase Model EOF Summary:")
print(f"Average first EOF step: {base_eof_df['first_eof_step'].mean():.1f}")
print(f"Average stable EOF step: {base_eof_df['stable_eof_step'].mean():.1f}")
print(f"Average EOF frequency: {base_eof_df['eof_frequency'].mean():.3f}")
print(f"Seeds with EOF loops: {(base_eof_df['eof_loops'] > 0).sum()}/{len(base_eof_df)}")
print(f"Seeds reaching stable EOF: {base_eof_df['stable_eof_step'].notna().sum()}/{len(base_eof_df)}")


Base Model EOF Analysis:
   seed  first_eof_step  stable_eof_step  total_eof_steps  eof_frequency
0     0               0                0                3            1.0
1     1               0                0               20            1.0
2     2               0                0               20            1.0
3     3               0                0               20            1.0
4     4               0                0               20            1.0
5     5               0                0               20            1.0
6     6               0                0               20            1.0
7     7               0                0               20            1.0
8     8               0                0               20            1.0
9     9               0                0               20            1.0

Base Model EOF Summary:
Average first EOF step: 0.0
Average stable EOF step: 0.0
Average EOF frequency: 1.000
Seeds with EOF loops: 20/20
Seeds reaching stable EOF: 20/20


## Instruct Model Analysis: Terminal Behavior Classification


In [9]:
def classify_terminal_behavior(steps: List[str]) -> Dict:
    """Classify the terminal behavior of instruct model.
    
    Returns:
        Dict with classification and metrics
    """
    if not steps:
        return {'classification': 'empty', 'confidence': 0.0, 'steady_state_step': None}
    
    # Strip the "> EOF by user" artifacts and get clean content
    clean_steps = []
    for step in steps:
        # Remove the EOF artifact at the end
        clean_step = re.sub(r'\s*>\s*EOF by user\s*$', '', step).strip()
        if clean_step:  # Only keep non-empty steps
            clean_steps.append(clean_step)
    
    if not clean_steps:
        return {'classification': 'empty', 'confidence': 0.0, 'steady_state_step': None}
    
    # Get the last few clean steps for terminal behavior analysis
    last_steps = clean_steps[-3:] if len(clean_steps) >= 3 else clean_steps
    last_text = ' '.join(last_steps).lower()
    
    # Classification patterns - looking at actual conversation content
    patterns = {
        'EOF-collapse': [
            r'conversation.*ended.*prematurely',
            r'input.*output.*reached.*end',
            r'eof.*short for.*end of file',
            r'unexpectedly.*terminated',
            r'end.*of.*file.*indicates'
        ],
        'Polite-close': [
            r'have a great day',
            r'thank you',
            r'goodbye',
            r'farewell',
            r'see you later',
            r'take care',
            r'best regards',
            r'until next time',
            r'look forward to.*next',
            r'stay curious',
            r'free to start a new conversation',
            r'start a new conversation anytime'
        ],
        'Self-repair': [
            r'i think i misunderstood',
            r'let me clarify',
            r'i may have been confused',
            r'my apologies',
            r'i made an error',
            r'let me correct',
            r'i should have',
            r'actually.*i think',
            r'let me rephrase',
            r'i misspoke'
        ],
        'Recursive empathy': [
            r'i understand how you feel',
            r'i can relate to that',
            r'that must be difficult',
            r'i hear what you\'re saying',
            r'i appreciate your perspective',
            r'that\'s a great question',
            r'i see what you mean',
            r'that makes sense',
            r'i can understand',
            r'that sounds'
        ],
        'Symbolic reappropriation': [
            r'clever.*eof.*message',
            r'mimicking.*eof',
            r'creative.*way.*eof',
            r'eof.*creative',
            r'playful.*eof',
            r'eof.*as.*signal',
            r'interpreting.*eof',
            r'eof.*as.*message',
            r'clever one.*eof',
            r'eof.*acknowledge'
        ]
    }
    
    # Score each classification
    scores = {}
    for category, pattern_list in patterns.items():
        score = 0
        for pattern in pattern_list:
            matches = len(re.findall(pattern, last_text))
            score += matches
        scores[category] = score
    
    # Find the best classification
    if all(score == 0 for score in scores.values()):
        classification = 'unclassified'
        confidence = 0.0
    else:
        classification = max(scores, key=scores.get)
        total_matches = sum(scores.values())
        confidence = scores[classification] / total_matches if total_matches > 0 else 0.0
    
    # Estimate steady-state step (when behavior becomes repetitive)
    steady_state_step = estimate_steady_state(steps)
    
    return {
        'classification': classification,
        'confidence': confidence,
        'scores': scores,
        'steady_state_step': steady_state_step
    }

def analyze_initial_behavior(steps: List[str]) -> Dict:
    """Analyze the initial behavior pattern after the first EOF.
    
    Returns:
        Dict with initial behavior classification and metrics
    """
    if not steps:
        return {'initial_classification': 'empty', 'initial_confidence': 0.0}
    
    # Strip EOF artifacts and get clean content
    clean_steps = []
    for step in steps:
        clean_step = re.sub(r'\s*>\s*EOF by user\s*$', '', step).strip()
        if clean_step:
            clean_steps.append(clean_step)
    
    if len(clean_steps) < 2:
        return {'initial_classification': 'insufficient', 'initial_confidence': 0.0}
    
    # Look at the first 2-3 clean steps for initial behavior
    initial_steps = clean_steps[:3] if len(clean_steps) >= 3 else clean_steps
    initial_text = ' '.join(initial_steps).lower()
    
    # Initial behavior patterns (what it does right after EOF)
    initial_patterns = {
        'immediate_polite': [
            r'hello',
            r'hi there',
            r'how can i help',
            r'what can i do',
            r'how may i assist',
            r'i\'m here to help'
        ],
        'eof_explanation': [
            r'eof.*short for',
            r'end of file',
            r'conversation.*ended',
            r'input.*output.*reached.*end',
            r'unexpectedly.*terminated'
        ],
        'confused_response': [
            r'i\'m not sure',
            r'i don\'t understand',
            r'what do you mean',
            r'could you clarify',
            r'i think i misunderstood'
        ],
        'creative_interpretation': [
            r'clever',
            r'creative',
            r'playful',
            r'interesting',
            r'well played'
        ],
        'immediate_goodbye': [
            r'have a great day',
            r'goodbye',
            r'thank you',
            r'farewell',
            r'see you later'
        ]
    }
    
    # Score each initial classification
    initial_scores = {}
    for category, pattern_list in initial_patterns.items():
        score = 0
        for pattern in pattern_list:
            matches = len(re.findall(pattern, initial_text))
            score += matches
        initial_scores[category] = score
    
    # Find the best initial classification
    if all(score == 0 for score in initial_scores.values()):
        initial_classification = 'unclassified'
        initial_confidence = 0.0
    else:
        initial_classification = max(initial_scores, key=initial_scores.get)
        total_matches = sum(initial_scores.values())
        initial_confidence = initial_scores[initial_classification] / total_matches if total_matches > 0 else 0.0
    
    return {
        'initial_classification': initial_classification,
        'initial_confidence': initial_confidence,
        'initial_scores': initial_scores
    }

def estimate_steady_state(steps: List[str]) -> Optional[int]:
    """Find when the conversation first transitions to its final terminal behavior pattern."""
    if len(steps) < 2:
        return None
    
    # Strip EOF artifacts first
    clean_steps = []
    for step in steps:
        clean_step = re.sub(r'\s*>\s*EOF by user\s*$', '', step).strip()
        if clean_step:
            clean_steps.append(clean_step)
    
    if len(clean_steps) < 2:
        return None
    
    # Look for the first occurrence of terminal behavior patterns
    terminal_patterns = {
        'polite_close': [
            r'have a great day',
            r'thank you',
            r'goodbye',
            r'farewell',
            r'see you later',
            r'take care',
            r'best regards',
            r'until next time',
            r'free to start a new conversation',
            r'nice chatting with you'
        ],
        'symbolic_reappropriation': [
            r'clever.*eof',
            r'mimicking.*eof',
            r'creative.*eof',
            r'playful.*eof',
            r'well played.*eof',
            r'eof.*creative',
            r'eof.*signal',
            r'eof.*message'
        ],
        'self_repair': [
            r'i think i misunderstood',
            r'let me clarify',
            r'my apologies',
            r'i made an error',
            r'let me correct'
        ],
        'recursive_empathy': [
            r'i understand how you feel',
            r'i can relate to that',
            r'i appreciate your perspective',
            r'that\'s a great question'
        ],
        'eof_collapse': [
            r'conversation.*ended.*prematurely',
            r'input.*output.*reached.*end',
            r'eof.*short for.*end of file'
        ]
    }
    
    # Find the first step that shows terminal behavior
    for i, step in enumerate(clean_steps):
        step_lower = step.lower()
        
        for category, patterns in terminal_patterns.items():
            for pattern in patterns:
                if re.search(pattern, step_lower):
                    return i  # Return the first occurrence of terminal behavior
    
    # If no clear terminal pattern found, return None
    return None

# Analyze instruct model behaviors (both initial and terminal)
instruct_behavior_analysis = []

for _, row in instruct_df.iterrows():
    # Get both initial and terminal behavior analysis
    initial_analysis = analyze_initial_behavior(row['steps'])
    terminal_analysis = classify_terminal_behavior(row['steps'])
    
    # Combine the analyses
    combined_analysis = {
        'seed': row['seed'],
        'initial_classification': initial_analysis['initial_classification'],
        'initial_confidence': initial_analysis['initial_confidence'],
        'terminal_classification': terminal_analysis['classification'],
        'terminal_confidence': terminal_analysis['confidence'],
        'steady_state_step': terminal_analysis['steady_state_step']
    }
    
    instruct_behavior_analysis.append(combined_analysis)

instruct_behavior_df = pd.DataFrame(instruct_behavior_analysis)

print("Instruct Model Behavior Analysis:")
print(instruct_behavior_df[['seed', 'initial_classification', 'terminal_classification', 'steady_state_step']].head(10))

# Summary statistics
print(f"\nInitial Behavior Distribution:")
initial_counts = instruct_behavior_df['initial_classification'].value_counts()
print(initial_counts)

print(f"\nTerminal Behavior Distribution:")
terminal_counts = instruct_behavior_df['terminal_classification'].value_counts()
print(terminal_counts)

print(f"\nAverage steady-state step: {instruct_behavior_df['steady_state_step'].mean():.1f}")
print(f"Average initial confidence: {instruct_behavior_df['initial_confidence'].mean():.3f}")
print(f"Average terminal confidence: {instruct_behavior_df['terminal_confidence'].mean():.3f}")

# Transition matrix (initial -> terminal)
print(f"\nTransition Matrix (Initial -> Terminal):")
transition_matrix = instruct_behavior_df.groupby(['initial_classification', 'terminal_classification']).size().unstack(fill_value=0)
print(transition_matrix)


Instruct Model Behavior Analysis:
   seed initial_classification   terminal_classification  steady_state_step
0     0      immediate_goodbye  Symbolic reappropriation                  0
1     1      immediate_goodbye              Polite-close                  0
2     2      immediate_goodbye              Polite-close                  0
3     3      immediate_goodbye              Polite-close                  0
4     4        eof_explanation              unclassified                  0
5     5        eof_explanation              Polite-close                  1
6     6      immediate_goodbye  Symbolic reappropriation                  0
7     7      immediate_goodbye              unclassified                  0
8     8      immediate_goodbye              Polite-close                  0
9     9      immediate_goodbye              unclassified                  0

Initial Behavior Distribution:
initial_classification
immediate_goodbye          13
immediate_polite            3
eof_explanation